# Step 1 — Tubularity: OOF + Structure Tensor (Guided Weighting)

```
output/stack_preprocessed.tif   (← step0_preprocess)
  ↓  OOF  → I_OOF(x) + v_OOF(x)
  ↓  Structure Tensor (per-scale) → v_ST(x)
  ↓  Guided Weighting
       C_align(x) = |<v_OOF, v_ST>|
       W(x) = I_OOF(x) · C_align(x)
  → output/tubularity.npz
```

## Guided Weighting

OOF response와 Structure Tensor는 같은 tube axis를 독립적으로 추정합니다.
두 방법의 방향이 일치할 때만 response를 강화하고, 불일치(노이즈/artifact)는 억제합니다.

| 항목 | 의미 |
|------|------|
| `v_OOF` | Q 행렬 최대 eigenvalue의 eigenvector (tube axis, 구 표면 flux 기반) |
| `v_ST`  | J 행렬 최소 eigenvalue의 eigenvector (tube axis, gradient outer product 기반) |
| `C_align` | 두 방향의 cosine similarity: 1=완전 일치, 0=직교 |
| `W`       | 최종 tubularity: OOF response × alignment confidence |

**Scale matching**: ST sigma_d = radius / 2 (OOF와 동일 scale에서 방향 추정)

In [ ]:
# ── Config ──────────────────────────────────────────────────
import os; os.makedirs('output', exist_ok=True)

PREPROCESSED_TIF  = 'output/FN1_01/stack_preprocessed.tif'
PREPROCESSED_META = 'output/FN1_01/preprocess_meta.npz'

# 입력 다운샘플링 (1 = 원본, 2 = 2× 축소 → 8× 빠름, 가는 axon 소실)
INPUT_DOWNSAMPLE = 1

# OOF
TUBE_RADIUS_MIN_UM  = 0.10   # 생물학적 하한 (µm) — voxel보다 작으면 자동 상향
TUBE_RADIUS_MAX_UM  = 2.0    # 생물학적 상한 (µm) — thick dendrite까지, soma shell 감지 방지
TUBE_RADIUS_MIN_VOX = 1.0    # 최소 반경 (voxel 단위) — 해상도 무관 floor
N_RADII             = 8
N_SPHERE_PTS        = 150
SLAB_SIZE           = 48

# Structure Tensor: sigma_d = radius / ST_SIGMA_RATIO  (per-scale matching)
ST_SIGMA_RATIO = 2.0    # sigma_d = radius / ST_SIGMA_RATIO
ST_SIGMA_INTEG = 2.0    # rho: outer-product smoothing (fixed)

# Guided Weighting
BETA              = 1.0   # alignment enhancement: W = T_raw × (1 + β·align)
LAMBDA_BLOB       = 0.3   # blob (LoG) weight added to OOF
RIDGE_FILL_KERNEL = 3     # max-pool kernel for hollow→solid tube filling

OUT_TIF = 'output/FN1_01/T_combined.tif'
OUT_NPZ = 'output/FN1_01/tubularity.npz'

In [ ]:
# ── Imports ─────────────────────────────────────────────────
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage
import gc, time, warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#111111',
    'axes.facecolor':   '#111111',
    'axes.titlecolor':  'white',
    'text.color':       'white',
})

def show_mip(vol, title='MIP', cmap='gray', vmin=None, vmax=None, ar=1.0):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    axes[0].imshow(vol.max(axis=0), aspect='equal', **kw); axes[0].set_title('XY')
    axes[1].imshow(vol.max(axis=1), aspect=ar,     **kw); axes[1].set_title('XZ')
    axes[2].imshow(vol.max(axis=2), aspect=ar,     **kw); axes[2].set_title('YZ')
    for ax in axes: ax.axis('off')
    fig.suptitle(title, fontsize=13); plt.show()

print('Imports ready.')

In [ ]:
# ── Load from step0 ──────────────────────────────────────────
meta      = np.load(PREPROCESSED_META)
voxel_iso = float(meta['voxel_iso'])
stack_iso = tifffile.imread(PREPROCESSED_TIF).astype(np.float32)

if INPUT_DOWNSAMPLE > 1:
    from scipy.ndimage import zoom
    stack_iso = zoom(stack_iso, 1.0 / INPUT_DOWNSAMPLE, order=1).astype(np.float32)
    voxel_iso = voxel_iso * INPUT_DOWNSAMPLE
    print(f'Downsampled {INPUT_DOWNSAMPLE}×: {stack_iso.shape}  voxel_iso={voxel_iso:.4f} µm')

print(f'Loaded   : {PREPROCESSED_TIF}')
print(f'Shape    : {stack_iso.shape}  (Z, Y, X)')
print(f'Range    : {stack_iso.min():.4f} - {stack_iso.max():.4f}')
print(f'Memory   : {stack_iso.nbytes / 1e9:.2f} GB')
print(f'voxel_iso: {voxel_iso:.4f} um')

NZ, NY, NX = stack_iso.shape

# ── Radius range: clamp min to TUBE_RADIUS_MIN_VOX ───────────
r_min = max(TUBE_RADIUS_MIN_UM, TUBE_RADIUS_MIN_VOX * voxel_iso)
r_max = TUBE_RADIUS_MAX_UM
if r_min > TUBE_RADIUS_MIN_UM:
    print(f'  radius min: {TUBE_RADIUS_MIN_UM:.3f} µm → {r_min:.3f} µm  '
          f'(floor {TUBE_RADIUS_MIN_VOX} vox × {voxel_iso:.4f} µm/vox)')
else:
    print(f'  radius min: {r_min:.3f} µm  ({r_min/voxel_iso:.2f} vox)')
print(f'  radius max: {r_max:.3f} µm  ({r_max/voxel_iso:.2f} vox)')

radii_vox = np.logspace(
    np.log10(r_min / voxel_iso),
    np.log10(r_max / voxel_iso),
    N_RADII,
).astype(np.float32)
OVERLAP = int(np.ceil(float(radii_vox[-1]) * 3))

print(f'Radii (vox): {radii_vox.round(2)}')
print(f'Overlap    : {OVERLAP} vox')

show_mip(stack_iso, title='Input (from step0)', ar=1.0)

In [ ]:
# ── Scale summary ─────────────────────────────────────────────
print(f'{"#":>3}  {"r (vox)":>8}  {"r (µm)":>8}  {"sigma_d (vox)":>13}  {"sigma_i (vox)":>13}')
print('-' * 55)
for i, r in enumerate(radii_vox):
    sigma_d = max(0.5, float(r) / ST_SIGMA_RATIO)
    print(f'{i+1:>3}  {r:>8.3f}  {r*voxel_iso:>8.3f}  {sigma_d:>13.3f}  {ST_SIGMA_INTEG:>13.1f}')
print('-' * 55)
print(f'OVERLAP = {OVERLAP} vox  |  voxel_iso = {voxel_iso:.4f} µm/vox')

In [ ]:
# ── Device detection ─────────────────────────────────────────
try:
    import torch
    import torch.nn.functional as F
    if torch.backends.mps.is_available():
        DEVICE  = torch.device('mps')
        USE_GPU = True
        print(f'Backend : MPS (Apple Metal)  PyTorch {torch.__version__}')
    elif torch.cuda.is_available():
        DEVICE  = torch.device('cuda')
        USE_GPU = True
        print(f'Backend : CUDA  PyTorch {torch.__version__}')
    else:
        DEVICE  = torch.device('cpu')
        USE_GPU = False
        print('No GPU - CPU fallback')
except ImportError:
    DEVICE = None; USE_GPU = False
    print('PyTorch not installed - CPU fallback')

---
## OOF + Structure Tensor Implementation

In [ ]:
# ── Sphere sampling (Fibonacci) ──────────────────────────────
def fibonacci_sphere(n: int) -> np.ndarray:
    """(n, 3) unit vectors in ZYX order, uniform on sphere."""
    golden = (1 + np.sqrt(5)) / 2
    i      = np.arange(n, dtype=np.float64)
    theta  = np.arccos(np.clip(1 - 2*(i + 0.5)/n, -1, 1))
    phi    = 2 * np.pi * i / golden
    return np.stack([
        np.cos(theta),
        np.sin(theta) * np.sin(phi),
        np.sin(theta) * np.cos(phi),
    ], axis=1).astype(np.float32)  # (N, 3)  ZYX

SPHERE_DIRS = fibonacci_sphere(N_SPHERE_PTS)
print(f'Sphere dirs: {SPHERE_DIRS.shape}')

In [ ]:
# ── Shared: 3x3 symmetric Cardano eigensolver (CPU) ──────────
# Used by both OOF and Structure Tensor.
# Returns sorted eigenvalues (ascending) and optionally eigenvectors.

def _cardano_eigs_cpu(Mzz, Myy, Mxx, Mzy, Mzx, Myx):
    """Eigenvalues of symmetric 3x3, sorted ascending. Returns (lam1, lam2, lam3)."""
    p1  = Mzy**2 + Mzx**2 + Myx**2
    q   = (Mzz + Myy + Mxx) / 3.0
    p2  = (Mzz-q)**2 + (Myy-q)**2 + (Mxx-q)**2 + 2*p1
    p   = np.sqrt(np.maximum(p2/6, 0))
    ps  = np.where(p > 1e-12, p, 1.0)
    B00 = np.where(p>1e-12, (Mzz-q)/ps, 0.)
    B11 = np.where(p>1e-12, (Myy-q)/ps, 0.)
    B22 = np.where(p>1e-12, (Mxx-q)/ps, 0.)
    B01 = np.where(p>1e-12, Mzy/ps, 0.)
    B02 = np.where(p>1e-12, Mzx/ps, 0.)
    B12 = np.where(p>1e-12, Myx/ps, 0.)
    r   = np.clip((B00*(B11*B22-B12**2) - B01*(B01*B22-B12*B02)
                   + B02*(B01*B12-B11*B02)) / 2, -1, 1)
    phi = np.arccos(r) / 3
    tp  = 2*np.pi/3
    e1  = (q + 2*p*np.cos(phi)).astype(np.float32)
    e2  = (q + 2*p*np.cos(phi + tp)).astype(np.float32)
    e3  = (q + 2*p*np.cos(phi + 2*tp)).astype(np.float32)
    eigs    = np.stack([e1, e2, e3], axis=-1)
    eigs    = np.sort(eigs, axis=-1)
    return eigs[...,0], eigs[...,1], eigs[...,2]   # lam1 <= lam2 <= lam3


def _eigvec_cpu(Mzz, Myy, Mxx, Mzy, Mzx, Myx, lam):
    """Eigenvector of eigenvalue lam via cross-product of two deflated rows."""
    r0 = np.stack([Mzz - lam, Mzy, Mzx], axis=-1)
    r1 = np.stack([Mzy, Myy - lam, Myx], axis=-1)
    v  = np.stack([
        r0[...,1]*r1[...,2] - r0[...,2]*r1[...,1],
        r0[...,2]*r1[...,0] - r0[...,0]*r1[...,2],
        r0[...,0]*r1[...,1] - r0[...,1]*r1[...,0],
    ], axis=-1)
    return (v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-8)).astype(np.float32)


print('Cardano eigensolver ready.')

In [ ]:
# ── Shared: 3x3 symmetric Cardano eigensolver (GPU) ──────────
if USE_GPU:
    def _cardano_eigs_gpu(Mzz, Myy, Mxx, Mzy, Mzx, Myx):
        p1  = Mzy**2 + Mzx**2 + Myx**2
        q   = (Mzz + Myy + Mxx) / 3.0
        p2  = (Mzz-q)**2 + (Myy-q)**2 + (Mxx-q)**2 + 2*p1
        p   = torch.sqrt(torch.clamp(p2/6, min=0))
        ps  = p.clamp(min=1e-12)
        B00 = (Mzz-q)/ps; B11 = (Myy-q)/ps; B22 = (Mxx-q)/ps
        B01 = Mzy/ps;      B02 = Mzx/ps;      B12 = Myx/ps
        r   = torch.clamp((B00*(B11*B22-B12**2) - B01*(B01*B22-B12*B02)
                           + B02*(B01*B12-B11*B02)) / 2, -1, 1)
        phi = torch.acos(r) / 3
        tp  = float(2*np.pi/3)
        e1  = q + 2*p*torch.cos(phi)
        e2  = q + 2*p*torch.cos(phi + tp)
        e3  = q + 2*p*torch.cos(phi + 2*tp)
        lam, _ = torch.sort(torch.stack([e1,e2,e3], dim=-1), dim=-1)
        return lam[...,0], lam[...,1], lam[...,2]  # ascending

    def _eigvec_gpu(Mzz, Myy, Mxx, Mzy, Mzx, Myx, lam):
        r0 = torch.stack([Mzz - lam, Mzy, Mzx], dim=-1)
        r1 = torch.stack([Mzy, Myy - lam, Myx], dim=-1)
        v  = torch.stack([
            r0[...,1]*r1[...,2] - r0[...,2]*r1[...,1],
            r0[...,2]*r1[...,0] - r0[...,0]*r1[...,2],
            r0[...,0]*r1[...,1] - r0[...,1]*r1[...,0],
        ], dim=-1)
        return v / (torch.linalg.norm(v, dim=-1, keepdim=True).clamp(min=1e-8))

    print('Cardano eigensolver (GPU) ready.')

In [ ]:
# ── GPU helpers: 1D separable Gaussian ──────────────────────
if USE_GPU:
    def _gk(sigma, order):
        r = int(np.ceil(4.0 * sigma))
        x = torch.arange(-r, r+1, dtype=torch.float32, device=DEVICE)
        g = torch.exp(-0.5 * (x / sigma) ** 2)
        g = g / g.sum()
        if order == 1: g = (-x / sigma**2) * g
        if order == 2: g = ((x**2 / sigma**4) - 1.0 / sigma**2) * g
        return g

    def _c1d(v, k, axis):
        pad = k.shape[0] // 2
        Z, Y, X = v.shape
        kv = k.view(1, 1, -1)
        if axis == 0:
            u = v.permute(1,2,0).contiguous().reshape(Y*X, 1, Z)
            u = torch.nn.functional.pad(u, (pad, pad), mode='replicate')
            return torch.nn.functional.conv1d(u, kv).reshape(Y, X, Z).permute(2,0,1)
        elif axis == 1:
            u = v.permute(0,2,1).contiguous().reshape(Z*X, 1, Y)
            u = torch.nn.functional.pad(u, (pad, pad), mode='replicate')
            return torch.nn.functional.conv1d(u, kv).reshape(Z, X, Y).permute(0,2,1)
        else:
            u = v.contiguous().reshape(Z*Y, 1, X)
            u = torch.nn.functional.pad(u, (pad, pad), mode='replicate')
            return torch.nn.functional.conv1d(u, kv).reshape(Z, Y, X)

    print('GPU Gaussian helpers ready.')


In [ ]:
# ── Structure Tensor: GPU ────────────────────────────────────
if USE_GPU:
    def structure_tensor_gpu(slab_t, sigma_d, sigma_i):
        sigma_d = max(0.5, sigma_d)
        k0d = _gk(sigma_d, 0); k1d = _gk(sigma_d, 1)
        k0i = _gk(sigma_i, 0)
        g0d = lambda v, ax: _c1d(v, k0d, ax)
        g1d = lambda v, ax: _c1d(v, k1d, ax)
        g0i = lambda v, ax: _c1d(v, k0i, ax)

        # Gaussian derivatives
        gz = g0d(g0d(g1d(slab_t, 0), 1), 2)
        gy = g0d(g1d(g0d(slab_t, 0), 1), 2)
        gx = g1d(g0d(g0d(slab_t, 0), 1), 2)

        # Smooth outer products (integration scale rho)
        sm = lambda v: g0i(g0i(g0i(v, 0), 1), 2)
        Jzz = sm(gz * gz); Jyy = sm(gy * gy); Jxx = sm(gx * gx)
        Jzy = sm(gz * gy); Jzx = sm(gz * gx); Jyx = sm(gy * gx)
        del gz, gy, gx

        lam1, _, _ = _cardano_eigs_gpu(Jzz, Jyy, Jxx, Jzy, Jzx, Jyx)
        v_st = _eigvec_gpu(Jzz, Jyy, Jxx, Jzy, Jzx, Jyx, lam1)
        del Jzz, Jyy, Jxx, Jzy, Jzx, Jyx, lam1
        return v_st  # (Z,Y,X,3) float32

    print('Structure tensor (GPU) ready.')


In [ ]:
# ── OOF: CPU ─────────────────────────────────────────────────
# Returns: (response, v_oof)
#   response : (Z,Y,X) float32  |lam1+lam2| when both negative
#   v_oof    : (Z,Y,X,3) float32  eigenvector of lam3 (tube axis)

from scipy.ndimage import map_coordinates

def oof_slab_cpu(slab, radius, dirs):
    Z, Y, X = slab.shape
    N       = dirs.shape[0]
    zg, yg, xg = np.mgrid[0:Z, 0:Y, 0:X]
    base = np.stack([zg.ravel(), yg.ravel(), xg.ravel()], axis=0).astype(np.float32)

    Q = np.zeros((6, Z*Y*X), dtype=np.float32)  # zz,yy,xx,zy,zx,yx
    for nz, ny, nx_ in dirs:
        off  = radius * np.array([[nz],[ny],[nx_]], dtype=np.float32)
        Ip   = map_coordinates(slab, base + off, order=1, mode='nearest', prefilter=False)
        Im   = map_coordinates(slab, base - off, order=1, mode='nearest', prefilter=False)
        gn   = (Ip - Im) / (2.0 * radius + 1e-8)
        Q[0] += gn * nz  * nz
        Q[1] += gn * ny  * ny
        Q[2] += gn * nx_ * nx_
        Q[3] += gn * nz  * ny
        Q[4] += gn * nz  * nx_
        Q[5] += gn * ny  * nx_
    Q /= N

    sh = (Z, Y, X)
    Qzz,Qyy,Qxx = Q[0].reshape(sh),Q[1].reshape(sh),Q[2].reshape(sh)
    Qzy,Qzx,Qyx = Q[3].reshape(sh),Q[4].reshape(sh),Q[5].reshape(sh)

    lam1, lam2, lam3 = _cardano_eigs_cpu(Qzz,Qyy,Qxx,Qzy,Qzx,Qyx)

    # tubularity: |lam1 + lam2| when both negative
    both_neg = (lam1 < 0) & (lam2 < 0)
    response = np.where(both_neg, np.abs(lam1 + lam2), 0.0).astype(np.float32)

    # tube axis = eigenvector of lam3 (largest, near-zero along tube)
    v_oof = _eigvec_cpu(Qzz,Qyy,Qxx,Qzy,Qzx,Qyx, lam3)

    return response, v_oof

In [ ]:
# ── OOF: GPU ─────────────────────────────────────────────────
if USE_GPU:
    def oof_slab_gpu(slab_t, radius, dirs_t):
        Z, Y, X = slab_t.shape
        N       = dirs_t.shape[0]
        vol5    = slab_t.unsqueeze(0).unsqueeze(0)  # (1,1,Z,Y,X)

        gz = torch.linspace(-1, 1, Z, device=DEVICE)
        gy = torch.linspace(-1, 1, Y, device=DEVICE)
        gx = torch.linspace(-1, 1, X, device=DEVICE)
        gz3, gy3, gx3 = torch.meshgrid(gz, gy, gx, indexing='ij')
        base_grid = torch.stack([gx3, gy3, gz3], dim=-1)  # (Z,Y,X,3) in xyz

        dz_n = radius * 2.0 / max(Z-1, 1)
        dy_n = radius * 2.0 / max(Y-1, 1)
        dx_n = radius * 2.0 / max(X-1, 1)

        Q = torch.zeros(6, Z*Y*X, dtype=torch.float32, device=DEVICE)

        for k in range(N):
            nz, ny, nx_ = dirs_t[k,0], dirs_t[k,1], dirs_t[k,2]
            off = torch.stack([nx_*dx_n, ny*dy_n, nz*dz_n])  # xyz order
            Ip  = F.grid_sample(vol5, (base_grid + off).unsqueeze(0),
                                mode='bilinear', padding_mode='border',
                                align_corners=True).squeeze().ravel()
            Im  = F.grid_sample(vol5, (base_grid - off).unsqueeze(0),
                                mode='bilinear', padding_mode='border',
                                align_corners=True).squeeze().ravel()
            gn  = (Ip - Im) / (2.0 * radius + 1e-8)
            Q[0] += gn * nz  * nz
            Q[1] += gn * ny  * ny
            Q[2] += gn * nx_ * nx_
            Q[3] += gn * nz  * ny
            Q[4] += gn * nz  * nx_
            Q[5] += gn * ny  * nx_
        Q /= N

        sh = (Z, Y, X)
        Qzz,Qyy,Qxx = Q[0].reshape(sh),Q[1].reshape(sh),Q[2].reshape(sh)
        Qzy,Qzx,Qyx = Q[3].reshape(sh),Q[4].reshape(sh),Q[5].reshape(sh)

        lam1, lam2, lam3 = _cardano_eigs_gpu(Qzz,Qyy,Qxx,Qzy,Qzx,Qyx)

        both_neg = (lam1 < 0) & (lam2 < 0)
        response = torch.where(both_neg, torch.abs(lam1 + lam2),
                               torch.zeros_like(lam1)).float()

        v_oof = _eigvec_gpu(Qzz,Qyy,Qxx,Qzy,Qzx,Qyx, lam3)

        return response, v_oof

In [ ]:
# ── Structure Tensor: orientation at given scale ─────────────
# sigma_d matched to OOF radius: sigma_d = radius / ST_SIGMA_RATIO
# Returns v_st: (Z,Y,X,3) float32  eigenvector of lam_min (tube axis)

def structure_tensor_cpu(slab, sigma_d, sigma_i):
    sigma_d = max(0.5, sigma_d)
    gz = ndimage.gaussian_filter(slab, sigma=sigma_d, order=(1,0,0), output=np.float32)
    gy = ndimage.gaussian_filter(slab, sigma=sigma_d, order=(0,1,0), output=np.float32)
    gx = ndimage.gaussian_filter(slab, sigma=sigma_d, order=(0,0,1), output=np.float32)

    Jzz = ndimage.gaussian_filter(gz*gz, sigma=sigma_i).astype(np.float32)
    Jyy = ndimage.gaussian_filter(gy*gy, sigma=sigma_i).astype(np.float32)
    Jxx = ndimage.gaussian_filter(gx*gx, sigma=sigma_i).astype(np.float32)
    Jzy = ndimage.gaussian_filter(gz*gy, sigma=sigma_i).astype(np.float32)
    Jzx = ndimage.gaussian_filter(gz*gx, sigma=sigma_i).astype(np.float32)
    Jyx = ndimage.gaussian_filter(gy*gx, sigma=sigma_i).astype(np.float32)
    del gz, gy, gx; gc.collect()

    lam1, _, _ = _cardano_eigs_cpu(Jzz,Jyy,Jxx,Jzy,Jzx,Jyx)
    # tube axis = eigenvector of lam_min (smallest eigenvalue of J)
    v_st = _eigvec_cpu(Jzz,Jyy,Jxx,Jzy,Jzx,Jyx, lam1)
    return v_st


print('Structure tensor (CPU) ready.')

In [ ]:
# ── Blob response (LoG): soma + tube center detection ────────
# Bright blob: all Hessian eigenvalues negative → -Tr(H) > 0
# Complements OOF (edge-dominant) by filling soma and tube centers

def blob_response_cpu(slab, sigma):
    sigma = max(0.5, sigma)
    Hzz = ndimage.gaussian_filter(slab, sigma=sigma, order=(2,0,0)).astype(np.float32)
    Hyy = ndimage.gaussian_filter(slab, sigma=sigma, order=(0,2,0)).astype(np.float32)
    Hxx = ndimage.gaussian_filter(slab, sigma=sigma, order=(0,0,2)).astype(np.float32)
    return np.maximum(0.0, -(Hzz + Hyy + Hxx))

if USE_GPU:
    def blob_response_gpu(slab_t, sigma):
        sigma = max(0.5, sigma)
        k2 = _gk(sigma, 2)
        k0 = _gk(sigma, 0)
        Hzz = _c1d(_c1d(_c1d(slab_t, k2, 0), k0, 1), k0, 2)
        Hyy = _c1d(_c1d(_c1d(slab_t, k0, 0), k2, 1), k0, 2)
        Hxx = _c1d(_c1d(_c1d(slab_t, k0, 0), k0, 1), k2, 2)
        return torch.clamp(-(Hzz + Hyy + Hxx), min=0.0)

print('Blob response ready.')

---
## Guided Weighting Loop

In [ ]:
# ── Main loop: Guided Weighting (enhanced) ────────────────────
W_combined   = np.zeros((NZ, NY, NX), np.float32)
I_OOF_raw    = np.zeros((NZ, NY, NX), np.float32)
orient_field = np.zeros((NZ, NY, NX, 3), np.float16)
scale_idx    = np.zeros((NZ, NY, NX), np.uint8)
radius_map   = np.zeros((NZ, NY, NX), np.float32)

if USE_GPU:
    dirs_t = torch.from_numpy(SPHERE_DIRS).to(DEVICE)

n_slabs = int(np.ceil(NZ / SLAB_SIZE))
backend = 'GPU' if USE_GPU else 'CPU'
print(f'Stack: {stack_iso.shape}  Slabs: {n_slabs}  Backend: {backend}')
print(f'Radii: {N_RADII}  Sphere pts: {N_SPHERE_PTS}')
t0 = time.time()

for slab_i, z0 in enumerate(range(0, NZ, SLAB_SIZE)):
    z1     = min(z0 + SLAB_SIZE, NZ)
    z0p    = max(0, z0 - OVERLAP)
    z1p    = min(NZ, z1 + OVERLAP)
    core_s = z0 - z0p
    core_e = z1 - z0p
    slab   = stack_iso[z0p:z1p]
    cZ     = z1 - z0

    best_W    = np.zeros((cZ, NY, NX), np.float32)
    best_IOOF = np.zeros((cZ, NY, NX), np.float32)
    best_si   = np.zeros((cZ, NY, NX), np.uint8)
    best_v    = np.zeros((cZ, NY, NX, 3), np.float32)

    for ri, radius in enumerate(radii_vox):
        sigma_d = float(radius) / ST_SIGMA_RATIO

        if USE_GPU:
            slab_t         = torch.from_numpy(slab).to(DEVICE)
            resp_t, voof_t = oof_slab_gpu(slab_t, float(radius), dirs_t)
            resp_c         = resp_t[core_s:core_e].cpu().numpy()
            voof_c         = voof_t[core_s:core_e].cpu().numpy()
            del resp_t, voof_t
        else:
            resp_full, voof_full = oof_slab_cpu(slab, float(radius), SPHERE_DIRS)
            resp_c  = resp_full[core_s:core_e]
            voof_c  = voof_full[core_s:core_e]
            del resp_full, voof_full

        if USE_GPU:
            core_blob = slab_t[core_s:core_e]
            blob_t    = blob_response_gpu(core_blob, float(radius))
            blob_c    = blob_t.cpu().numpy()
            del core_blob, blob_t
        else:
            blob_c = blob_response_cpu(slab[core_s:core_e], float(radius))
        blob_c *= float(radius) ** 2

        if USE_GPU:
            core_t   = slab_t[core_s:core_e]
            vst_t    = structure_tensor_gpu(core_t, sigma_d, ST_SIGMA_INTEG)
            vst_full = vst_t.cpu().numpy()
            del core_t, vst_t, slab_t
            if DEVICE.type == 'mps': torch.mps.empty_cache()
        else:
            vst_full = structure_tensor_cpu(slab[core_s:core_e], sigma_d, ST_SIGMA_INTEG)

        c_align = np.abs((voof_c * vst_full).sum(axis=-1))
        T_raw   = resp_c + LAMBDA_BLOB * blob_c
        W       = T_raw * (1.0 + BETA * c_align)

        improved = W > best_W
        best_W[improved]    = W[improved]
        best_IOOF[improved] = resp_c[improved]
        best_si[improved]   = ri
        best_v[improved]    = voof_c[improved]

        del resp_c, voof_c, vst_full, c_align, blob_c, T_raw, W, improved; gc.collect()

    W_combined[z0:z1]   = best_W
    I_OOF_raw[z0:z1]    = best_IOOF
    orient_field[z0:z1] = best_v.astype(np.float16)
    scale_idx[z0:z1]    = best_si

    del best_W, best_IOOF, best_si, best_v; gc.collect()
    print(f'  slab {slab_i+1}/{n_slabs}  z={z0}-{z1}  {time.time()-t0:.0f}s', end='\r')

# ── Ridge filling: max-pool hollow tubes → solid ridges ─────────
from scipy.ndimage import maximum_filter
W_combined = maximum_filter(W_combined, size=RIDGE_FILL_KERNEL)

W_combined /= (W_combined.max() + 1e-10)
I_OOF_raw  /= (I_OOF_raw.max()  + 1e-10)
radius_map  = radii_vox[scale_idx] * voxel_iso

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'W_combined : max={W_combined.max():.4f}  mean={W_combined.mean():.6f}')
print(f'I_OOF_raw  : max={I_OOF_raw.max():.4f}  mean={I_OOF_raw.mean():.6f}')

In [ ]:
# ── CHECK: W vs I_OOF ────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for row, (vol, lbl) in enumerate([(I_OOF_raw, 'I_OOF (raw)'), (W_combined, 'W = I_OOF * C_align')]):
    for col, axis in enumerate([0, 1, 2]):
        axes[row, col].imshow(vol.max(axis=axis), cmap='hot', vmin=0, vmax=1)
        axes[row, col].set_title(f'{lbl} - {["XY","XZ","YZ"][col]}')
        axes[row, col].axis('off')
    axes[row, 3].hist(vol[vol > 0.02].ravel(), bins=80, color='steelblue', density=True)
    axes[row, 3].set_title(f'{lbl} histogram')

plt.suptitle('Guided Weighting: OOF vs W')
plt.tight_layout(); plt.show()

# C_align statistics (derived)
mask = I_OOF_raw > 0.01
c_mean = (W_combined[mask] / (I_OOF_raw[mask] + 1e-10)).mean()
print(f'Mean C_align (where I_OOF > 0.01): {c_mean:.3f}')
print(f'W > 0.05 : {(W_combined>0.05).mean()*100:.1f}%')
print(f'W > 0.10 : {(W_combined>0.10).mean()*100:.1f}%')

In [ ]:
  # ── Blend: I_OOF + W ───────────────────────────────────────── 
alphas = [0.0, 0.3, 0.7, 1]
p999_oof = np.percentile(I_OOF_raw[I_OOF_raw > 0], 99.9)
p999_w   = np.percentile(W_combined[W_combined > 0], 99.9)
I_OOF_norm = np.clip(I_OOF_raw / p999_oof, 0, 1)
W_norm     = np.clip(W_combined / p999_w,   0, 1)

fig, axes = plt.subplots(len(alphas), 3, figsize=(16, 4*len(alphas)))
  for i, a in enumerate(alphas):
      T = a * I_OOF_norm + (1 - a) * W_norm
      for col in range(3):
          axes[i, col].imshow(T.max(axis=col), cmap='hot', vmin=0, vmax=1)
          axes[i, col].set_title(f'α={a}  {"XY XZ YZ".split()[col]}')
          axes[i, col].axis('off')
  plt.tight_layout(); plt.show()

  for a in alphas:
      T = a * I_OOF_norm + (1 - a) * W_norm
      print(f'α={a}  T>0.05: {(T>0.05).mean()*100:.2f}%  T>0.10: {(T>0.10).mean()*100:.2f}%')

In [ ]:
print(W_combined.max(), W_combined.mean())

In [ ]:
# ── Save ────────────────────────────────────────────────────
p999 = np.percentile(W_combined[W_combined > 0], 99.5) 
T_combined = np.clip(W_combined / p999, 0, 1)

tifffile.imwrite(OUT_TIF, T_combined)
np.savez(OUT_NPZ,
    T_combined   = T_combined,
    I_OOF_raw    = I_OOF_raw,
    orient_field = orient_field,
    radius_map   = radius_map,
    scale_idx    = scale_idx,
    radii        = radii_vox,
    voxel_iso    = np.float32(voxel_iso),
)
print(f'Saved: {OUT_NPZ}')
print(f'  T_combined (W)  {W_combined.shape} float32')
print(f'  I_OOF_raw       {I_OOF_raw.shape} float32')
print(f'  orient_field    {orient_field.shape} float16')
print(f'  radius_map      {radius_map.shape} float32')